# MambaIR — Google Colab eğitim not defteri

Bu not defterini Colab’a yükleyip sırayla çalıştırın.

1. **Runtime → Change runtime type → GPU** (T4/L4 vb.) seçin.
2. Kendi GitHub çatallamanızı (`fork`) kullanacaksanız aşağıdaki `REPO_URL` değerini güncelleyin.
3. Eğitim verilerini Google Drive’a koyup hücredeki yolları kendi klasör yapınıza göre düzenleyin.
### Colab'a nasıl bağlanırım?

- **`Colab: False` görüyorsanız** not defteri şu an **yerelde** (Cursor, VS Code, Jupyter) çalışıyordur; bu **beklenen** davranıştır.
- **`Colab: True` için** tarayıcıda [colab.research.google.com](https://colab.research.google.com) açın → **File → Upload notebook** ile bu `.ipynb` dosyasını yükleyin (veya repoyu GitHub'a koyup Colab'da **File → Open notebook → GitHub** ile açın) → **Runtime → Change runtime type → GPU** seçin → hücreleri **Colab sayfasında** çalıştırın.


## 1. Colab ortamı

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("Colab:", IN_COLAB)
if not IN_COLAB:
    print(
        "→ Yerelde çalışıyorsunuz; True görmek için not defterini https://colab.research.google.com üzerinde açın."
    )


Colab: True


In [3]:
import torch

assert torch.cuda.is_available(), "Runtime türünde GPU seçili değil."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

GPU: Tesla T4
PyTorch: 2.10.0+cu128


## 2. Ayarlar

- `REPO_URL`: Clone edilecek repo (geliştirmeleriniz kendi fork’ınızdaysa onu yazın).
- `REPO_ROOT`: Kodun duracağı klasör (Colab’da genelde `/content/MambaIR`).
- Veri yolları: DIV2K / DF2K yapınıza göre HR ve LR klasörleri.

In [10]:
from pathlib import Path

# --- GitHub ---
REPO_URL = "https://github.com/alperslmz/MambaIR-AS.git"  # Kendi fork: https://github.com/KULLANICI/MambaIR.git
REPO_BRANCH = "mambair"  # veya çalıştığınız dal

REPO_ROOT = Path("/content/MambaIR")

# --- Google Drive (veri setleri) ---
MOUNT_DRIVE = True  # Veriyi Drive’dan okuyacaksanız True

# Örnek: Drive’da "MambaIR_data/DIV2K_train_HR" gibi yapı
DRIVE_DATA = Path("/content/drive/MyDrive/IRdataset")
#GT_TRAIN = DRIVE_DATA / "DIV2K_train_HR"
#LQ_TRAIN = DRIVE_DATA / "DIV2K_train_LR_bicubic" / "X2"  # x2 için; x3/x4 ise X3, X4

# Doğrulama (ör. Set14); yoksa geçici olarak train alt klasörü vermeyin—küçük bir val seti hazırlayın
GT_VAL = DRIVE_DATA / "Set14" / "HR"
LQ_VAL = DRIVE_DATA / "Set14" / "LR_bicubic" / "X2"

## 3. Google Drive bağlama

In [11]:
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
else:
    print("Drive atlanıyor (Colab değil veya MOUNT_DRIVE=False).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 4. Repoyu indirme / güncelleme

In [1]:
# clone the MambaIR repo
!git clone https://github.com/alperslmz/MambaIR-AS.git
%cd MambaIR-AS

fatal: destination path 'MambaIR-AS' already exists and is not an empty directory.
/content/MambaIR-AS


In [2]:


# Set up the environment, it may takes 2min
!pip install torch==2.2.2+cu118 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached https://download.pytorch.org/whl/cu118/torch-2.2.2%2Bcu118-cp312-cp312-linux_x86_64.whl (819.1 MB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cuda_nvrtc_cu11-11.8.89-py3-none-manylinux1_x86_64.whl (23.2 MB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cuda_runtime_cu11-11.8.89-py3-none-manylinux1_x86_64.whl (875 kB)
  Using cached https://download.pytorch.org/whl/cu118/nvidia_cuda_cupti_cu11-11.8.87-py3-none-manylinux1_x86_64.whl (13.1 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 MB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 MB 14.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.2/128.2 MB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import subprocess
import sys

def install_with_live_logs(packages):
    cmd = [sys.executable, "-m", "pip", "install"] + packages

    print(f"\n🚀 Kurulum başlıyor:\n👉 {' '.join(packages)}\n")

    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="")  # canlı log

    process.wait()

    if process.returncode != 0:
        raise Exception("❌ Kurulum hata verdi!")

    print("\n✅ Kurulum tamamlandı!\n")


install_with_live_logs(["mamba-ssm", "causal-conv1d", "ninja"])


🚀 Kurulum başlıyor:
👉 mamba-ssm causal-conv1d ninja

  Using cached mamba_ssm-2.3.1.tar.gz (121 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached causal_conv1d-1.6.1.tar.gz (29 kB)
  Installing build dependencies: started


In [13]:
import subprocess
import sys
import shutil # Import shutil for rmtree


def sh(*args, cwd=None):
    print("$", " ".join(args))
    subprocess.check_call(args, cwd=cwd)


# Check if the REPO_ROOT exists
if REPO_ROOT.exists():
    # If it exists, check if it's a valid git repository
    if (REPO_ROOT / ".git").is_dir():
        print(f"Repo directory '{REPO_ROOT}' already exists and is a git repository. Updating...")
        sh("git", "-C", str(REPO_ROOT), "fetch", "--all")
        sh("git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH)
        sh("git", "-C", str(REPO_ROOT), "pull", "origin", REPO_BRANCH)
    else:
        # Directory exists but is NOT a git repository, or it's a file.
        # Remove it to ensure a clean clone.
        print(f"Repo path '{REPO_ROOT}' exists but is not a valid git repository (or is a file). Removing it to perform a fresh clone.")
        if REPO_ROOT.is_dir():
            shutil.rmtree(REPO_ROOT)
        else: # It's a file
            REPO_ROOT.unlink() # Delete the file
        sh("git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT))
else:
    # REPO_ROOT does not exist, perform a fresh clone
    print(f"Repo path '{REPO_ROOT}' does not exist. Cloning...")
    sh("git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT))

print("Repo:", REPO_ROOT)


Repo path '/content/MambaIR' does not exist. Cloning...
$ git clone --branch mambair https://github.com/alperslmz/MambaIR-AS.git /content/MambaIR
Repo: /content/MambaIR


## 5. Python bağımlılıkları ve `basicsr` kurulumu

Colab’daki PyTorch sürümüne uyum için `mamba-ssm` / `causal-conv1d` bazen ek deneme gerektirir; hata alırsanız [mamba](https://github.com/state-spaces/mamba/releases) ve [causal-conv1d](https://github.com/Dao-AILab/causal-conv1d/releases) üzerindeki tekerlek (wheel) sürümlerini PyTorch + CUDA sürümünüze göre seçebilirsiniz.

`setup.py` içindeki `requirements.txt` conda formatında olduğu için kurulum **`--no-deps`** ile yapılıyor; aşağıdaki paketler elle ekleniyor.

In [17]:
PIP_DEPS = [
    "addict",
    "einops",
    "future",
    "lmdb",
    "numpy",
    "opencv-python",
    "pillow",
    "pyyaml",
    "requests",
    "scikit-image",
    "scipy",
    "tensorboard",
    "timm",
    "tqdm",
    "yapf",
    "packaging",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PIP_DEPS])

0

In [22]:
import torch
import sys
import subprocess

# Helper to install pre-built wheels based on Torch and CUDA versions
def install_wheels():
    torch_version = torch.__version__.split('+')[0]
    cuda_version = torch.version.cuda.replace('.', '')
    python_version = f"cp{sys.version_info.major}{sys.version_info.minor}"

    print(f"Installing wheels for Torch {torch_version}, CUDA {cuda_version}, Python {python_version}")

    try:
        # Construct generic or specific wheel links. For Colab, we try to use the ones provided by the authors.
        # Note: If exact versions are missing, you may need to specify exact release URLs from GitHub.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "causal-conv1d", "--no-build-isolation"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "mamba-ssm", "--no-build-isolation"])
    except Exception as e:
        print(f"Failed to install with --no-build-isolation: {e}")
        print("Please manually specify the wheel URL from https://github.com/state-spaces/mamba/releases")

install_wheels()

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT), "--no-deps"]
)

print("Kurulum tamam.")


Installing wheels for Torch 2.10.0, CUDA 128, Python cp312


KeyboardInterrupt: 

## 6. Kurulum doğrulama

In [16]:
import basicsr

print("basicsr import OK")

ModuleNotFoundError: No module named 'basicsr'

## 7. Colab için opsiyon dosyası (Drive yolları + tek GPU)

Orijinal `.yml` dosyalarındaki mutlak yollar yerine yukarıda tanımlı `GT_TRAIN` vb. kullanılır. Batch ve worker değerleri Colab belleğine göre düşürülmüştür; VRAM yetmezse `BATCH_SIZE` değerini azaltın.

In [ ]:
import yaml

BASE_YML = REPO_ROOT / "options/train/mambairv2/train_MambaIRv2_lightSR_x2.yml"
COLAB_YML = REPO_ROOT / "options/train/colab_generated_lightSR_x2.yml"

BATCH_SIZE = 32  # T4’te gerekirse 2 yapın
NUM_WORKERS = 2

assert GT_TRAIN.is_dir(), f"Bulunamadı: {GT_TRAIN}"
assert LQ_TRAIN.is_dir(), f"Bulunamadı: {LQ_TRAIN}"
assert GT_VAL.is_dir(), f"Bulunamadı: {GT_VAL}"
assert LQ_VAL.is_dir(), f"Bulunamadı: {LQ_VAL}"

with open(BASE_YML, "r", encoding="utf-8") as f:
    opt = yaml.load(f, Loader=yaml.FullLoader)

opt["name"] = opt.get("name", "colab") + "_colab"
opt["num_gpu"] = 1
opt["datasets"]["train"]["dataroot_gt"] = [str(GT_TRAIN)]
opt["datasets"]["train"]["dataroot_lq"] = [str(LQ_TRAIN)]
opt["datasets"]["train"]["batch_size_per_gpu"] = BATCH_SIZE
opt["datasets"]["train"]["num_worker_per_gpu"] = NUM_WORKERS

for k in opt["datasets"]:
    if k.startswith("val"):
        opt["datasets"][k]["dataroot_gt"] = str(GT_VAL)
        opt["datasets"][k]["dataroot_lq"] = str(LQ_VAL)

with open(COLAB_YML, "w", encoding="utf-8") as f:
    yaml.dump(opt, f, default_flow_style=False, allow_unicode=True, sort_keys=False)

print("Yazıldı:", COLAB_YML)

## 8. Eğitimi başlatma

Log ve ağırlıklar `experiments/` altında oluşur (`REPO_ROOT` içinde). Oturum kapanınca `/content` silinir; çıktıları **Drive’a kopyalayın** veya repo kökünü Drive üzerinde tutacak şekilde `git clone` hedefini değiştirin.

In [ ]:
cmd = [
    sys.executable,
    str(REPO_ROOT / "basicsr/train.py"),
    "-opt",
    str(COLAB_YML),
    "--launcher",
    "none",
]

print(" ".join(cmd))
# Eğitimi başlat:
subprocess.check_call(cmd, cwd=str(REPO_ROOT))